In [12]:
!git clone https://github.com/vivek28n/Medical-RAG-Hallucination-Detection.git

Cloning into 'Medical-RAG-Hallucination-Detection'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 44 (delta 19), reused 15 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 1.31 MiB | 23.96 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [13]:
%cd /content/Medical-RAG-Hallucination-Detection

/content/Medical-RAG-Hallucination-Detection


In [14]:
import os

print("Dataset:", os.path.exists("dataset"))
print("Raw:", os.path.exists("dataset/raw"))
print("Files:", os.listdir("dataset/raw"))

Dataset: True
Raw: True
Files: ['niddk_guiding_principles_diabetes.pdf.pdf']


In [15]:
!git pull origin main

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 4 (delta 1), reused 4 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 394 bytes | 394.00 KiB/s, done.
From https://github.com/vivek28n/Medical-RAG-Hallucination-Detection
 * branch            main       -> FETCH_HEAD
   febb88a..4b32342  main       -> origin/main
Updating febb88a..4b32342
Fast-forward
 ...abetes.pdf.pdf => niddk_guiding_principles_diabetes.pdf} | Bin
 1 file changed, 0 insertions(+), 0 deletions(-)
 rename dataset/raw/{niddk_guiding_principles_diabetes.pdf.pdf => niddk_guiding_principles_diabetes.pdf} (100%)


In [16]:
import os
print(os.listdir("dataset/raw"))

['niddk_guiding_principles_diabetes.pdf']


In [20]:
!pip install -q pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 64.2 MB/s eta 0:00:00


In [22]:
import os

print(os.getcwd())
print(os.listdir("dataset/raw"))

/content/Medical-RAG-Hallucination-Detection
['niddk_guiding_principles_diabetes.pdf']


In [26]:
import fitz

pdf_path = "dataset/raw/niddk_guiding_principles_diabetes.pdf"

doc = fitz.open(pdf_path)

print("Number of pages:", len(doc))

Number of pages: 83


In [27]:
text = ""

for page in doc:
    text += page.get_text()

print("Characters extracted:", len(text))
print("\nFirst 3000 characters:\n")
print(text[:3000])

Characters extracted: 201724

First 3000 characters:

1
Guiding Principles
for the Care of People with or at Risk for Diabetes
2
Supporting Organizations
The Guiding Principles for the Care of People with or at Risk for Diabetes was produced by the 
National Diabetes Education Program (NDEP),* a federally funded program sponsored by the 
U.S. Department of Health and Human Services’ National Institutes of Health and Centers for 
Disease Control and Prevention. NDEP’s partnership network includes more than 200 partners 
working together to improve the treatment and outcomes for people with diabetes, promote early 
diagnosis, and prevent or delay the onset of type 2 diabetes. The following organizations support 
the use of the Guiding Principles for the Care of People with or at Risk for Diabetes:
• Academy of Nutrition and Dietetics
• American Academy of Family Physicians
• American Academy of Physician Assistants
• American Association of Clinical Endocrinologists
• American Associatio

In [28]:
print("Total characters:", len(text))
print("Total words:", len(text.split()))
print("First 1000 characters:")
print(text[:1000])

Total characters: 201724
Total words: 28529
First 1000 characters:
1
Guiding Principles
for the Care of People with or at Risk for Diabetes
2
Supporting Organizations
The Guiding Principles for the Care of People with or at Risk for Diabetes was produced by the 
National Diabetes Education Program (NDEP),* a federally funded program sponsored by the 
U.S. Department of Health and Human Services’ National Institutes of Health and Centers for 
Disease Control and Prevention. NDEP’s partnership network includes more than 200 partners 
working together to improve the treatment and outcomes for people with diabetes, promote early 
diagnosis, and prevent or delay the onset of type 2 diabetes. The following organizations support 
the use of the Guiding Principles for the Care of People with or at Risk for Diabetes:
• Academy of Nutrition and Dietetics
• American Academy of Family Physicians
• American Academy of Physician Assistants
• American Association of Clinical Endocrinologists
• Americ

In [29]:
pages = []

for page_number, page in enumerate(doc, start=1):
    page_text = page.get_text().strip()

    pages.append({
        "page": page_number,
        "text": page_text
    })

print("Total pages processed:", len(pages))

Total pages processed: 83


In [30]:
print("Page number:", pages[0]["page"])
print("Characters:", len(pages[0]["text"]))
print("\nPage text:\n")
print(pages[0]["text"][:2000])

Page number: 1
Characters: 72

Page text:

1
Guiding Principles
for the Care of People with or at Risk for Diabetes


In [31]:
import re

def clean_text(text):
    # Multiple spaces ko single space
    text = re.sub(r"[ \t]+", " ", text)

    # Multiple blank lines ko single newline
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    # Har line ke beginning/end ke extra spaces
    text = "\n".join(line.strip() for line in text.splitlines())

    return text.strip()


for page in pages:
    page["clean_text"] = clean_text(page["text"])

print("Cleaning completed.")

Cleaning completed.


In [32]:
print("Original characters:", len(pages[0]["text"]))
print("Cleaned characters:", len(pages[0]["clean_text"]))

print("\nCleaned page preview:\n")
print(pages[0]["clean_text"][:2000])

Original characters: 72
Cleaned characters: 72

Cleaned page preview:

1
Guiding Principles
for the Care of People with or at Risk for Diabetes


In [34]:
!pip install -q langchain-text-splitters

In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

print("Text splitter ready!")

Text splitter ready!


In [36]:
chunks = []

for page in pages:
    page_chunks = splitter.split_text(page["clean_text"])

    for chunk_id, chunk in enumerate(page_chunks):
        chunks.append({
            "chunk_id": f"page_{page['page']}_chunk_{chunk_id}",
            "page": page["page"],
            "text": chunk
        })

print("Total chunks:", len(chunks))

Total chunks: 277


In [37]:
print("Chunk ID:", chunks[0]["chunk_id"])
print("Page:", chunks[0]["page"])
print("Characters:", len(chunks[0]["text"]))
print("\nChunk:\n")
print(chunks[0]["text"])

Chunk ID: page_1_chunk_0
Page: 1
Characters: 72

Chunk:

1
Guiding Principles
for the Care of People with or at Risk for Diabetes


In [38]:
print("Chunk ID:", chunks[100]["chunk_id"])
print("Page:", chunks[100]["page"])
print("Characters:", len(chunks[100]["text"]))
print("\nChunk:\n")
print(chunks[100]["text"])

Chunk ID: page_33_chunk_0
Page: 33
Characters: 959

Chunk:

33
• Use of non-nutritive sweeteners can reduce overall calorie and carbohydrate intake if
substituted for caloric sweeteners without compensatory increase from other dietary sources.
• Treatment of mild hypoglycemia (plasma glucose < 70 mg/dL) requires ingestion of 15 to
20 grams of glucose through carbohydrate-containing foods or glucose tablets. Because
protein appears to increase insulin response without increasing glucose, people with diabetes
should not use carbohydrate sources high in protein to prevent or treat hypoglycemia.
Encourage physical activity 4-6,9
Regular physical activity helps improve insulin sensitivity and glycemic control, positively affects
lipids and blood pressure, assists with weight maintenance, and is associated with reduced risk
for CVD.4-6 Regular physical activity also can improve psychological well-being, health-related
quality of life, and depression in individuals with type 2 diabetes, among

In [39]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [40]:
test_embedding = embedding_model.encode(chunks[0]["text"])

print("Embedding type:", type(test_embedding))
print("Embedding dimensions:", len(test_embedding))

Embedding type: <class 'numpy.ndarray'>
Embedding dimensions: 384


In [41]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

print("Total embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape[1])

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Total embeddings: 277
Embedding dimensions: 384


In [43]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 33.1 MB/s eta 0:00:00


In [44]:
import faiss
import numpy as np

embedding_array = np.array(embeddings).astype("float32")

print("Embedding shape:", embedding_array.shape)

Embedding shape: (277, 384)


In [45]:
dimension = embedding_array.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embedding_array)

print("Total vectors in FAISS:", index.ntotal)

Total vectors in FAISS: 277


In [46]:
query = "What are the risk factors for diabetes?"

query_embedding = embedding_model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")

print("Query embedding shape:", query_embedding.shape)

Query embedding shape: (1, 384)


In [47]:
distances, indices = index.search(query_embedding, k=5)

print("Retrieved indices:", indices[0])
print("Distances:", distances[0])

Retrieved indices: [  7  23   8 250   0]
Distances: [0.6567528  0.6770457  0.7259885  0.7448276  0.74978125]


In [48]:
for rank, idx in enumerate(indices[0], start=1):
    print(f"\n--- Result {rank} ---")
    print("Chunk ID:", chunks[idx]["chunk_id"])
    print("Page:", chunks[idx]["page"])
    print("Distance:", distances[0][rank - 1])
    print("\n", chunks[idx]["text"][:1000])


--- Result 1 ---
Chunk ID: page_5_chunk_0
Page: 5
Distance: 0.6567528

 5
INTRODUCTION
The diabetes problem
Today, 30.3 million people (9.4 percent of the U.S. population) have diabetes, including 7.2 million
who are undiagnosed.1 A major cause of blindness, renal failure, and amputation, diabetes
also increases the risk of cardiovascular disease, cancer, and dementia and more than doubles
individual health care costs.2 The total estimated cost of diagnosed diabetes in 2017 was $327
billion, including $237 billion in direct medical costs and $90 billion in reduced productivity.2
Another 84.1 million Americans (33.9 percent of adults) have glucose levels that are higher than
normal but not high enough to be characterized as diabetes.1 Because persons with these glucose
levels are at increased risk of developing type 2 diabetes, this condition is termed prediabetes
by the Centers for Disease Control and Prevention (CDC) and other organizations.
Proper nutrition and physical activity are